In [83]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd

In [84]:
urls =[ "https://www.bbc.com/news/articles/crlp991nw41o",
        "https://www.bbc.com/future/article/20260519-google-tackles-attempts-to-hack-its-ai-results",
        "https://www.bbc.com/news/articles/cvgz1ynq1nqo",
        "https://www.bbc.com/news/articles/crep3v8vzglo",
        "https://www.bbc.com/news/articles/c4g04qkqlk2o",
        "https://www.bbc.com/future/article/20260505-how-to-use-ai-without-turning-your-brain-to-mush",
        "https://www.bbc.com/news/articles/c0q2gkj97eko",
        "https://www.bbc.com/news/articles/c62jp09p0l4o",
        "https://www.bbc.com/future/article/20251218-how-ai-can-teach-us-to-really-listen",
        "https://www.bbc.com/news/articles/cvgzr935jvmo"]
len(urls)

10

In [85]:
import nltk
nltk.download('punkt')
from nltk import RegexpTokenizer

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [86]:
def get_article_text(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=15)
    soup = bs(r.text, "html.parser")

    if soup.title:
        title = soup.title.get_text(strip=True)
    else:
        title = "No Title"

    para = soup.find_all("p")
    l = []
    for p in para:
        l.append(p.get_text(" ", strip=True))
    text = " ".join(l)
    raw_text = text

    text = text.lower()

    tokenizer = RegexpTokenizer(r'\w+')
    tokens = tokenizer.tokenize(text)

    c = len(tokens)

    return {"url": url, "title": title, "text": raw_text,"tokens": tokens,"token_count": c}

In [87]:
rows = []
for url in urls:
    try:
        article = get_article_text(url)
        rows.append(article)
        print("Done:", article["title"])
    except Exception as e:
        rows.append({"url": url, "title": "ERROR", "text": "", "tokens": [], "token_count": 0})
        print("Failed:", url)
        print("Error:", e)

Done: Why the AI industry is the real winner of the Musk-Altman trial
Done: Google's AI is being manipulated. The search giant is quietly fighting back
Done: Google to release first smart glasses since Google Glass flop
Done: Standard Chartered to cut thousands of roles as AI use increases
Done: Samsung strike on hold as workers push for AI bonus
Done: 'Think outside the bots': How to stop AI from turning your brain to mush
Done: Robo-top: The machines that could make your next t-shirt
Done: The fight against foreign developers buying Caribbean beaches
Done: They hear, but do they care? What AI can teach us about listening better
Done: University of Lincoln AI stand wins award at Chelsea Flower Show


In [88]:
df = pd.DataFrame(rows)
df.head()

,url,title,text,tokens,token_count
0,https://www.bbc.com/news/articles/crlp991nw41o,Why the AI industry is the real winner of the ...,It is not only OpenAI but the AI race itself t...,"[it, is, not, only, openai, but, the, ai, race...",943
1,https://www.bbc.com/future/article/20260519-go...,Google's AI is being manipulated. The search g...,A BBC investigation revealed a simple way AI c...,"[a, bbc, investigation, revealed, a, simple, w...",1859
2,https://www.bbc.com/news/articles/cvgz1ynq1nqo,Google to release first smart glasses since Go...,More than a decade after its famed Google Glas...,"[more, than, a, decade, after, its, famed, goo...",669
3,https://www.bbc.com/news/articles/crep3v8vzglo,Standard Chartered to cut thousands of roles a...,Banking giant Standard Chartered has become th...,"[banking, giant, standard, chartered, has, bec...",473
4,https://www.bbc.com/news/articles/c4g04qkqlk2o,Samsung strike on hold as workers push for AI ...,The largest union at Samsung Electronics has s...,"[the, largest, union, at, samsung, electronics...",789


In [89]:
df.to_csv("data.csv", index=False)

In [90]:
from sentence_transformers import SentenceTransformer as st
import numpy as np

model = st("sentence-transformers/all-MiniLM-L6-v2")

texts = df["text"].fillna("").tolist()
embeddings = model.encode(texts, show_progress_bar=True)

print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(10, 384)


In [91]:
emb_df = pd.DataFrame(embeddings)
emb_df.to_csv("embeddings.csv", index=False)